# Analiza dialogowa scenariusza: z jakim ładunkiem emocjonalnym?

## Wersja studencka — moduł 3: EMOCJE

W poprzednim module wydobyłeś kwestie dialogowe i zmierzyłeś, *kto* mówi i *ile*. Teraz pytamy o *jak* — z jakim ładunkiem emocjonalnym. Do oceny emocji użyjemy modelu językowego (OpenAI), który przypisze każdej wypowiedzi walencję (od −1 do +1) oraz emocję dyskretną.

Efektem końcowym będą:
1. wczytanie kwestii z `dialogi.csv` i danych o postaciach z `postacie.csv`,
2. konfiguracja połączenia z OpenAI API,
3. batchowa ocena emocji wszystkich kwestii,
4. łuk emocjonalny filmu (walencja wzdłuż scen),
5. wykres rozkładu emocji według płci postaci,
6. zapis pliku `emocje.csv` do dalszej analizy.

## Jak pracować z tym notatnikiem

1. Upewnij się, że masz w tym samym środowisku pliki `dialogi.csv` i `postacie.csv` z NB1.
2. Skopiuj prompt z bieżącego kroku do modelu AI — **ale najpierw przeczytaj go i uzupełnij luki**.
3. Wklej otrzymany kod do pustej komórki pod promptem.
4. Uruchom kod i porównaj z sekcją **Po uruchomieniu powinieneś zobaczyć**.
5. Dopiero po uzyskaniu poprawnego wyniku przejdź dalej.

**Uwaga do kroku z API:** w Etapie 2 będziesz potrzebować klucza do OpenAI API. Klucz jest prywatny — nie wklejaj go na stałe do komórek.

## Twoja kolej: uzupełniaj prompty

W NB1 czytałeś gotowe prompty i uczyłeś się rozumieć ich 7-sekcyjną strukturę. Teraz kilka promptów ma **luki oznaczone `[UZUPEŁNIJ: …]`**, które musisz wypełnić.

**Jak to działa:**
Zamiast tekstu w markdown, prompty żyją teraz w zmiennej `moj_prompt` w komórce kodu.
1. Przeczytaj sekcję **Cel i sens analityczny** — opisuje, *co* chcemy osiągnąć.
2. Przeczytaj sekcję **Po uruchomieniu powinieneś zobaczyć** — mówi, *jak* powinien wyglądać wynik.
3. **Edytuj zmienną `moj_prompt`** — wypełnij każdą lukę `[UZUPEŁNIJ: …]`.
4. **Uruchom komórkę** — asercja sprawdzi, czy wszystkie luki są wypełnione i prompt ma odpowiednią treść.
5. Jeśli asercja przejdzie, skopiuj wypisany prompt do asystenta AI.

Zasada: luka `[UZUPEŁNIJ: …]` opisuje **co** powinieneś tam wpisać — ale Ty decydujesz, **jak** to sformulujesz.

---
## Etap 1/6 — Wczytanie danych wejściowych

Zaczynamy od danych z NB1.

In [ ]:
# === ETAP 1/6 · WCZYTANIE DANYCH ===
import os
brak = [f for f in ["dialogi.csv", "postacie.csv"] if not os.path.exists(f)]
if brak:
    raise FileNotFoundError(
        f"⛔ Brakuje plików: {brak}\n"
        "Pobierz je z NB1 (Etap 6) i wgraj do tego środowiska."
    )
print("📍 Etap 1/6 · Wczytanie danych. Pliki dialogi.csv i postacie.csv — znalezione ✓")

### Krok 1A. Wczytanie `dialogi.csv` i `postacie.csv`

#### Cel i sens analityczny

`dialogi.csv` zawiera po jednym wierszu na kwestię. `postacie.csv` zawiera podsumowanie postaci z płcią. Musimy załadować oba i sprawdzić, że wyglądają poprawnie.

#### Prompt dla modelu

```text
Kontekst:
Masz dwa pliki CSV z poprzedniego modułu analizy dialogowej.

Wejście:
Pliki `dialogi.csv` i `postacie.csv` w kodowaniu UTF-8.

Zadanie:
Wczytaj oba pliki do tabel i wyświetl krótki podgląd każdego z nich. Zachowaj wczytane tabele jako zmienne o nazwach `dialogi` i `postacie`.

Pokaż wynik:
- liczbę wierszy i nazwy kolumn każdego pliku,
- pierwsze 5 wierszy z `dialogi`,
- wszystkie wiersze z `postacie` (jest ich mało).

Warunek poprawności:
Kolumny powinny odpowiadać opisanemu formatowi. Jeśli nazwy kolumn są inne, wypisz to wprost.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się wczytać i z jakiego powodu.

Nie rób jeszcze:
Nie oceniaj emocji.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Liczbę wierszy i kolumny obu plików.
- Kilka pierwszych kwestii z `dialogi`.
- Tabelę postaci z kolumną płeć (K/M/?).

---
## Etap 2/6 — Konfiguracja modelu językowego

Instalujemy bibliotekę OpenAI, podłączamy klucz API i sprawdzamy, czy połączenie działa.

In [ ]:
# === ETAP 2/6 · KONFIGURACJA OPENAI ===
assert 'dialogi' in dir(), "⛔ Najpierw uruchom Etap 1 — brak tabeli `dialogi`."
print(f"📍 Etap 2/6 · Konfiguracja OpenAI. Wejście: `dialogi` ({len(dialogi):,} kwestii).")

### Krok 2A. Instalacja i klucz API

#### Cel i sens analityczny

Żeby wywołać OpenAI z kodu Pythona, potrzebna jest biblioteka `openai` i klucz API. Klucz wpisujemy przez `getpass` — tak, żeby nie pojawił się w historii komórek.

#### Prompt dla modelu

```text
Kontekst:
Pracujesz w Google Colab i chcesz wywołać OpenAI API z Pythona.

Wejście:
Brak — konfiguracja od zera.

Zadanie:
Zainstaluj pakiet `openai`. Następnie poproś użytkownika o wpisanie klucza API przez `getpass.getpass` i utwórz klienta: `client = openai.OpenAI(api_key=...)`. Na końcu wyślij jedno testowe żądanie do modelu `gpt-4o-mini` (krótka wiadomość: „Hello!") i wydrukuj odpowiedź, żeby potwierdzić że połączenie działa.

Pokaż wynik:
- komunikat, że instalacja się powiodła,
- pole do wpisania klucza (bez jego wyświetlania),
- krótka odpowiedź modelu na testowe zapytanie.

Warunek poprawności:
Model musi odpowiedzieć — nawet jednym zdaniem. Błąd 401 = problem z kluczem.

Jeśli wystąpi błąd:
Wyświetl treść błędu i podpowiedź: sprawdź, czy klucz jest aktywny i ma środki na koncie.

Nie rób jeszcze:
Nie oceniaj kwestii dialogowych.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie instalacji i pole do wpisania klucza API.
- Krótką odpowiedź modelu potwierdzającą działające połączenie.

---
## Etap 3/6 — Ocena emocji dialogów

Model językowy przeanalizuje każdą kwestię i przypisze jej:
- **walencję** — liczbę od −1 (bardzo negatywna) do +1 (bardzo pozytywna),
- **emocję dyskretną** — jedną z sześciu: `radość`, `smutek`, `złość`, `strach`, `zaskoczenie`, `neutralna`.

Wysyłamy kwestie **partiami** — żeby nie przekroczyć limitu żądań per minutę.

In [ ]:
# === ETAP 3/6 · OCENA EMOCJI ===
assert 'dialogi' in dir(), "⛔ Najpierw uruchom Etap 1 — brak tabeli `dialogi`."
print(f"📍 Etap 3/6 · Ocena emocji. Wejście: {len(dialogi):,} kwestii do oceny.")

### Krok 3A. Batchowa ocena emocji

#### Cel i sens analityczny

Ocena linia po linii (jedno żądanie = jedna kwestia) jest wolna i kosztowna. Batching — przesyłanie wielu kwestii w jednym żądaniu i odebranie odpowiedzi jako tablicy JSON — jest znacznie wydajniejszy.

#### Twoja kolej: uzupełnij i uruchom

> **Instrukcja dla modelu (środkowy blok między `---`) jest wbudowana na stałe — nie zmieniaj jej. Wypełnij dwie oznaczone luki `[UZUPEŁNIJ: …]`, potem uruchom.**

In [ ]:
# TWOJA KOLEJ: wypełnij każdą lukę, potem uruchom tę komórkę.
moj_prompt = """
Kontekst:
Masz tabelę kwestii dialogowych w zmiennej `dialogi`. Masz skonfigurowanego klienta OpenAI API (zmienna `client` — obiekt `openai.OpenAI`).

Wejście:
Tabela `dialogi` z kolumnami: numer sceny, postać, treść kwestii.

Zadanie:
Napisz funkcję przetwarzającą kwestie partiami po [UZUPEŁNIJ: ile kwestii na partię? — typowe wartości to 20–50]. Dla każdej partii wyślij do OpenAI jedno żądanie (`client.chat.completions.create`, model `gpt-4o-mini`) z poniższą instrukcją wbudowaną na stałe w kod:

---
Oceń emocje poniższych kwestii dialogowych ze scenariusza filmowego.
Dla każdej kwestii zwróć obiekt JSON z polami:
- "id": numer porządkowy kwestii w tej partii (0-indeksowany),
- "walencja": liczba od -1.0 (bardzo negatywna) do 1.0 (bardzo pozytywna),
- "emocja": jedna z: radość, smutek, złość, strach, zaskoczenie, neutralna.
Zwróć WYŁĄCZNIE tablicę JSON, bez żadnego dodatkowego tekstu ani formatowania markdown.
Kwestie do oceny:
[TUTAJ WSTAW KWESTIE W FORMACIE: "id | postać: treść kwestii"]
---

Po odebraniu odpowiedzi sparsuj JSON i dołącz wyniki do tabeli kwestii jako nowe kolumny `walencja` i `emocja`. Zachowaj wynik jako zmienną o nazwie `emocje`. Jeśli parsowanie partii się nie powiedzie, wypełnij brakujące wiersze: walencja=0, emocja="neutralna" i wypisz ostrzeżenie. Dodaj opóźnienie [UZUPEŁNIJ: ile sekund między partiami? — 1–2 sekundy wystarczają] między partiami.

Pokaż wynik:
- komunikaty o przetwarzaniu kolejnych partii (numer partii i postęp),
- po zakończeniu: łączną liczbę ocenionych kwestii,
- próbkę 10 wierszy: postać | treść (pierwsze 60 znaków) | walencja | emocja.

Warunek poprawności:
Liczba ocenionych kwestii = liczba wierszy w `dialogi`. Kolumna `emocja` zawiera wyłącznie wartości z listy: radość, smutek, złość, strach, zaskoczenie, neutralna.

Jeśli wystąpi błąd:
Wypisz, przy której partii wystąpił problem i pokaż surową odpowiedź modelu.

Nie rób jeszcze:
Nie rysuj wykresów.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Zostały nieuzupełnione luki [UZUPEŁNIJ… — wypełnij każdą."
assert len(moj_prompt.strip()) > 200, \
    "⛔ Prompt za krótki — wypełnij sekcje konkretną treścią o SWOIM filmie."
assert "emocje" in moj_prompt, \
    "⛔ Upewnij się, że Twój prompt mówi modelowi o zapisaniu wyniku w zmiennej `emocje`."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Komunikaty o przetwarzaniu kolejnych partii.
- Próbkę kwestii z przypisanymi walencjami i emocjami.

> **Wskazówka:** jeśli model zwraca JSON opakowany w znaczniki ` ```json ... ``` `, poproś go o poprawkę: *„usuń markdown z odpowiedzi — zwracaj tylko czystą tablicę JSON"*.

### Krok 3B. Podgląd i kontrola jakości

#### Cel i sens analityczny

Zanim narysujemy wykresy, sprawdzamy, czy rozkład emocji wygląda sensownie — czy model nie wpadł w schemat (np. wszystko „neutralna").

#### Twoja kolej: uzupełnij i uruchom

Przejrzyj dane w komórce podglądu powyżej. Wypełnij jedną lukę `[UZUPEŁNIJ: …]` w komórce kodu poniżej i uruchom ją — komórka sprawdzi i wypisze gotowy prompt.

In [ ]:
# TWOJA KOLEJ: wypełnij lukę [UZUPEŁNIJ: …], potem uruchom tę komórkę.
moj_prompt = """
Kontekst:
Masz tabelę kwestii z przypisanymi emocjami i walencjami w zmiennej `emocje`.

Wejście:
Tabela `emocje` z kolumnami widocznymi powyżej w komórce podglądu.

Zadanie:
Policz rozkład emocji: ile kwestii i jaki procent przypada na każdą kategorię. Pokaż statystyki opisowe walencji (min, mediana, średnia, max). Wypisz 3 kwestie z najwyższą walencją i 3 z najniższą (postać + treść + walencja).

Pokaż wynik:
- tabelę: emocja | liczba kwestii | procent,
- statystyki walencji (min, mediana, max),
- 3 kwestie najwyżej i 3 najniżej nacechowane emocjonalnie.

Warunek poprawności:
[UZUPEŁNIJ: po jakim progu uznasz, że rozkład jest zdegenerowany? — np. "jeśli jedna emocja ma ponad __% kwestii, wypisz ostrzeżenie"]

Jeśli wystąpi błąd:
Wyjaśnij, czy kolumna emocja jest pusta albo zawiera nieoczekiwane wartości.

Nie rób jeszcze:
Nie rysuj wykresów.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Wypełnij lukę [UZUPEŁNIJ: …] — podaj próg procentowy dla zdegenerowanego rozkładu."
assert len(moj_prompt.strip()) > 200, \
    "⛔ Prompt za krótki — wypełnij brakującą sekcję."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Tabelę: emocja | liczba kwestii | procent.
- Podstawowe statystyki walencji (min, mediana, max).
- 3 kwestie z najwyższą walencją i 3 z najniższą.

> **Jeśli rozkład jest zdegenerowany** (np. 90% „neutralna") — wróć do Kroku 3A i poproś model AI o zmodyfikowanie wbudowanej instrukcji oceny emocji, żeby była bardziej rozróżniająca.

---
## Etap 4/6 — Łuk emocjonalny filmu

Teraz patrzymy na emocje w czasie — jak zmienia się nastrój od pierwszej do ostatniej sceny. To tzw. *emotional arc*, znany z badań Reagana i in. (2016) oraz Vonneguta.

In [ ]:
# === ETAP 4/6 · ŁUK EMOCJONALNY ===
assert 'emocje' in dir(), "⛔ Najpierw uruchom Etap 3 — brak tabeli `emocje`."
print(f"📍 Etap 4/6 · Łuk emocjonalny. Wejście: `emocje` ({len(emocje):,} kwestii z walencją).")

In [ ]:
# HIPOTEZA — wpisz ZANIM narysujesz wykres
# Gdzie w Twoim filmie będzie najgłębsza dolina emocjonalna (najbardziej mroczna sekwencja)?
# Opisz scenę lub moment fabuły.
moja_hipoteza_4 = ""   # np. "środkowa część — scena konfrontacji głównych bohaterów"
assert moja_hipoteza_4.strip(), \
    "⛔ Najpierw opisz, gdzie spodziewasz się emocjonalnego dołka — zanim zobaczysz wykres."
print("✏️  Hipoteza zapisana. Teraz wygeneruj wykres łuku emocjonalnego.")

#### Twoja kolej: uzupełnij i uruchom

Przejrzyj dane w komórce podglądu powyżej. Wypełnij jedną lukę `[UZUPEŁNIJ: …]` w komórce kodu poniżej i uruchom.

In [ ]:
# TWOJA KOLEJ: wypełnij lukę [UZUPEŁNIJ: …], potem uruchom tę komórkę.
moj_prompt = """
Kontekst:
Masz tabelę kwestii z walencją i numerem sceny w zmiennej `emocje`.

Wejście:
Tabela `emocje` z kolumnami widocznymi powyżej w komórce podglądu. Potrzebujesz kolumny z numerem sceny i kolumny walencja.

Zadanie:
(1) Wylicz średnią walencję dla każdej sceny. (2) Wygładź wynik oknem przesuwnym o szerokości [UZUPEŁNIJ: ile scen? — mniejsze okno = więcej szczegółów, większe = płynniejszy przebieg; typowo 5–15]. (3) Narysuj wykres liniowy: oś X = numer sceny, oś Y = wygładzona walencja, pozioma linia zerowa wyróżniona kolorem.

Pokaż wynik:
- wykres liniowy z tytułem i etykietami osi,
- wyraźna pozioma linia y=0 (inna grubość lub kolor),
- numer sceny z najniższą wygładzoną walencją wypisany pod wykresem.

Warunek poprawności:
Wykres nie powinien być płaski — wypisz ostrzeżenie, jeśli odchylenie standardowe wygładzonej walencji jest mniejsze niż 0.05.

Jeśli wystąpi błąd:
Wyjaśnij, czy tabela nie ma kolumny walencja albo czy wszystkie numery scen są identyczne.

Nie rób jeszcze:
Nie rysuj wykresu emocja × płeć.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Wypełnij lukę [UZUPEŁNIJ: …] — podaj szerokość okna wygładzającego (liczbę scen)."
assert len(moj_prompt.strip()) > 200, \
    "⛔ Prompt za krótki — wypełnij brakującą sekcję."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

In [ ]:
# CHECKPOINT — uzupełnij na podstawie wykresu powyżej
numer_doliny = ""  # np. "45" — numer sceny z najgłębszą doliną (odczytaj z wykresu)
assert numer_doliny.strip().isdigit(), \
    "⛔ Wpisz numer sceny z najniższą walencją (odczytaj z wykresu)."
print(f"📊 Checkpoint: najgłębsza dolina emocjonalna — scena {numer_doliny}.")
print(f"Twoja hipoteza mówiła o: '{moja_hipoteza_4[:80]}'")
print("Porównaj z hipotezą powyżej.")

#### Pytanie interpretacyjne

Gdzie na wykresie są doliny i szczyty? Czy pokrywają się z ważnymi zwrotami fabuły, które pamiętasz z filmu? Vonnegut opisywał opowieści jako „kształty" — jaki kształt ma Twój film?

### Krok 4B. Rozkład emocji dyskretnych w czasie

#### Cel i sens analityczny

Walencja pyta *czy* scena jest nacechowana pozytywnie lub negatywnie. Emocje dyskretne mówią *jak* — złość i smutek to obie emocje negatywne, ale niosą inne znaczenie narracyjne. Ten wykres pokazuje, jak zmienia się proporcja poszczególnych emocji wzdłuż scenariusza.

In [ ]:
# PRZEGLĄD — uruchom przed uzupełnieniem prompta
assert 'emocje' in dir(), "⛔ Najpierw uruchom Etap 3."
_scene_col = next((c for c in emocje.columns
                   if any(k in c.lower() for k in ('scena', 'scene', 'numer', 'num'))), None)
if _scene_col:
    _n = emocje[_scene_col].nunique()
    print(f"📋 Liczba scen: {_n}")
    print(f"    → Grupowanie co  5 scen → ~{_n // 5:>2} słupków")
    print(f"    → Grupowanie co 10 scen → ~{_n // 10:>2} słupków")
    print(f"    → Grupowanie co 15 scen → ~{_n // 15:>2} słupków")
else:
    print(f"Kolumny dostępne: {list(emocje.columns)}")
_emocja_col = next((c for c in emocje.columns if 'emocj' in c.lower()), None)
if _emocja_col:
    print(f"\nEmocje w zbiorze: {list(emocje[_emocja_col].unique())}")

#### Wypełnij jedną lukę

Prompt jest prawie gotowy. Znajdź `[UZUPEŁNIJ: …]` i wpisz, co ile scen grupować dane — to decyzja o rozdzielczości wykresu (patrz podgląd powyżej).

In [ ]:
# TWOJA KOLEJ: uzupełnij jedną lukę [UZUPEŁNIJ: …], potem uruchom.
moj_prompt = """
Kontekst:
Masz tabelę kwestii z przypisanymi emocjami dyskretycznymi i numerami scen w zmiennej `emocje`. Kolumna emocja zawiera 6 kategorii: radość/smutek/złość/strach/zaskoczenie/neutralna.

Wejście:
Tabela `emocje` — sprawdź jej kolumny poleceniem `emocje.columns` i dopasuj nazwy.

Zadanie:
(1) Pogrupuj sceny w przedziały po [UZUPEŁNIJ: co ile scen? — patrz podgląd powyżej, typowo 5–15] scen. (2) Dla każdego przedziału policz, jaki procent kwestii przypada na każdą z 6 emocji. (3) Narysuj skumulowany wykres słupkowy (stacked bar): oś X = przedział scen, oś Y = procent (0–100%), każda emocja = inny kolor. (4) Dodaj tytuł „Rozkład emocji w czasie" i legendę z nazwami emocji.

Pokaż wynik:
- skumulowany wykres słupkowy z tytułem i etykietami osi,
- legenda 6 emocji z kolorami.

Warunek poprawności:
Dla każdego słupka suma procentów = 100%. Każda emocja ma unikalny kolor.

Jeśli wystąpi błąd:
Sprawdź dostępne wartości emocji: `emocje['nazwa_kolumny'].unique()`.

Nie rób jeszcze:
Nie analizuj emocji per postać.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Pozostała luka [UZUPEŁNIJ: …] — wpisz, co ile scen grupujesz dane."
assert "emocje" in moj_prompt, \
    "⛔ Prompt musi odwoływać się do zmiennej `emocje`."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Skumulowany wykres słupkowy z 6 emocjami w różnych kolorach.
- Widoczne zmiany proporcji emocji między różnymi częściami scenariusza.

> **Pytanie interpretacyjne:** Czy emocja dominująca zmienia się między początkiem a końcem? W którym fragmencie pojawia się najwięcej złości? Kiedy najwięcej strachu?

### Krok 4C. Łuk emocjonalny wybranych postaci

#### Cel i sens analityczny

Ta sama scena, różne postacie — różne emocje. Antagonista może przeżywać radość dokładnie tam, gdzie protagonista przeżywa strach. Ten wykres zestawia trajektorie walencji trzech wybranych postaci wzdłuż całego scenariusza.

In [ ]:
# PRZEGLĄD — uruchom przed uzupełnieniem prompta, żeby wybrać 3 postacie
assert 'emocje' in dir(), "⛔ Najpierw uruchom Etap 3."
_char_col = next((c for c in emocje.columns
                  if any(k in c.lower() for k in ('postać', 'postac', 'character', 'speaker', 'bohater'))), None)
if _char_col:
    print(f"📋 Postaci z największą liczbą kwestii (kolumna: '{_char_col}'):\n")
    print(emocje.groupby(_char_col).size().sort_values(ascending=False).head(15).to_string())
    print("\n→ Wybierz 3 postacie i wpisz ich nazwy DOKŁADNIE jak powyżej.")
else:
    print(f"Kolumny dostępne: {list(emocje.columns)}")
    print("Sprawdź, która kolumna zawiera nazwy postaci.")

#### Wypełnij jedną lukę

Prompt jest prawie gotowy. Znajdź `[UZUPEŁNIJ: …]` i wpisz nazwy trzech postaci — **dokładnie tak**, jak figurują w danych powyżej (wielkie litery, spacje, pisownia).

In [ ]:
# TWOJA KOLEJ: uzupełnij jedną lukę [UZUPEŁNIJ: …], potem uruchom.
moj_prompt = """
Kontekst:
Masz tabelę kwestii z walencją (-1 do +1) i numerami scen w zmiennej `emocje`. Kolumny widoczne powyżej w komórce podglądu.

Wejście:
Tabela `emocje`. Trzy postacie do porównania: [UZUPEŁNIJ: wpisz 3 nazwy postaci oddzielone przecinkami — dokładnie jak w danych, np. "TYLER, THE NARRATOR, MARLA"].

Zadanie:
(1) Dla każdej z trzech postaci wybierz wiersze z `emocje`. (2) Oblicz średnią walencję na scenę dla każdej postaci. (3) Narysuj wykres liniowy z trzema liniami — jedna linia na postać, oś X = numer sceny, oś Y = walencja od -1 do +1. (4) Dodaj poziomą linię zerową (y=0). (5) Dodaj legendę z imionami postaci i tytuł „Łuk emocjonalny wybranych postaci".

Pokaż wynik:
- wykres liniowy z 3 liniami w różnych kolorach,
- legenda z imionami postaci,
- dla każdej postaci: liczba scen, w których wystąpiła (pod wykresem).

Warunek poprawności:
Na wykresie widać co najmniej 2 z 3 postaci. Jeśli postać ma mniej niż 5 scen, wypisz ostrzeżenie zamiast przerywać z błędem.

Jeśli wystąpi błąd:
Sprawdź dokładną pisownię imion: `emocje['kolumna_postaci'].unique()`.

Nie rób jeszcze:
Nie zapisuj pliku.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Pozostała luka [UZUPEŁNIJ: …] — wpisz nazwy 3 postaci oddzielone przecinkami."
assert "emocje" in moj_prompt, \
    "⛔ Prompt musi odwoływać się do zmiennej `emocje`."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Wykres z trzema liniami w różnych kolorach.
- Widoczne różnice w trajektoriach emocjonalnych między postaciami.

> **Pytanie interpretacyjne:** Czy „emocjonalne doliny" jednej postaci pokrywają się ze „szczytami" drugiej? Co to mówi o ich relacji w narracji?

---
## Etap 5/6 — Emocje według płci

Łączymy dwa wymiary: emocje i płeć. Pytamy, czy kobiety i mężczyźni w tym scenariuszu wyrażają inne emocje.

#### Twoja kolej: uzupełnij i uruchom

Przejrzyj dane w komórce podglądu powyżej. Wypełnij jedną lukę `[UZUPEŁNIJ: …]` w komórce kodu poniżej i uruchom.

In [ ]:
# TWOJA KOLEJ: wypełnij lukę [UZUPEŁNIJ: …], potem uruchom tę komórkę.
moj_prompt = """
Kontekst:
Masz tabelę kwestii z kolumną emocja w zmiennej `emocje`. Masz tabelę postaci z kolumną płeć (K/M/?) w zmiennej `postacie`. Nazwy kolumn widoczne powyżej w komórce podglądu.

Wejście:
Tabele `emocje` i `postacie` z kolumnami widocznymi powyżej.

Zadanie:
(1) Połącz tabele po nazwie postaci, żeby każda kwestia miała przypisaną płeć. (2) [UZUPEŁNIJ: co zrobić z wierszami o płci "?"? — pomiń je, wypisz osobno, czy potraktuj jako trzecią grupę na wykresie?]. (3) Policz, jaki procent kwestii z każdą emocją przypada na K i na M — normalizuj do 100% osobno dla każdej płci. (4) Narysuj grupowany wykres słupkowy: emocje na osi X, dwie grupy słupków (K i M) w różnych kolorach.

Pokaż wynik:
- grupowany wykres słupkowy z legendą K/M,
- wartości procentowe na słupkach,
- ostrzeżenie jeśli jedna z płci ma mniej niż 20 kwestii.

Warunek poprawności:
Procenty emocji dla K sumują się do 100%, tak samo dla M.

Jeśli wystąpi błąd:
Wyjaśnij, czy problem dotyczy łączenia tabel albo brakujących wartości płci.

Nie rób jeszcze:
Nie zapisuj pliku.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Wypełnij lukę [UZUPEŁNIJ: …] — zdecyduj, co zrobić z wierszami o płci '?'."
assert len(moj_prompt.strip()) > 200, \
    "⛔ Prompt za krótki — wypełnij brakującą sekcję."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Wykres porównujący rozkład emocji między kobietami i mężczyznami.
- Wartości procentowe (umożliwiające porównanie mimo różnej liczby kwestii).
- Wyraźne etykiety płci i kategorii emocji.

#### Pytanie interpretacyjne

Czy rozkład emocji różni się między płciami? Które emocje są „sfeminizowane" lub „zmaskulinizowane" w tym konkretnym filmie? Jak ten wynik ma się do Twoich intuicji z oglądania?

> **Zastrzeżenie metodologiczne:** emocja przypisana przez model to interpretacja tekstu dialogu — nie subiektywne odczucie postaci ani intencja autorska. Wynik mówi coś o *reprezentacji*, nie o *rzeczywistości*.

---
## Etap 6/6 — Eksport danych

Zapisujemy wyniki oceny emocji do pliku wejściowego dla NB3.

In [ ]:
# === ETAP 6/6 · EKSPORT ===
assert 'emocje' in dir(), "⛔ Najpierw uruchom Etap 3 — brak tabeli `emocje`."
print(f"📍 Etap 6/6 · Eksport. `emocje` ({len(emocje):,} wierszy) → pliki CSV.")

### Krok 6A. Zapis pliku `emocje.csv`

#### Cel i sens analityczny

`emocje.csv` to „wzbogacona" wersja `dialogi.csv` — zawiera kwestie z ocenami emocji. `emocje_postaci.csv` to emocjonalne portrety postaci. W NB3 dołączymy te dane do sieci postaci.

#### Prompt dla modelu

```text
Kontekst:
Masz tabelę kwestii z ocenionymi emocjami i walencjami w zmiennej `emocje`.

Wejście:
Tabela `emocje` z kolumnami: numer sceny, postać, treść kwestii, liczba słów, walencja, emocja.

Zadanie:
Zapisz tę tabelę do pliku `emocje.csv` w kodowaniu UTF-8. Następnie wygeneruj skrócony plik `emocje_postaci.csv`, w którym każda postać ma jeden wiersz z kolumnami: postać, dominująca emocja (najczęstsza), średnia walencja, liczba kwestii. Pokaż jak pobrać oba pliki z Colab.

Pokaż wynik:
- komunikat o zapisaniu obu plików z liczbą wierszy,
- 5 pierwszych wierszy `emocje_postaci.csv`,
- wskazówkę jak pobrać pliki.

Warunek poprawności:
Oba pliki niepuste. Liczba wierszy w `emocje.csv` = liczba kwestii.

Jeśli wystąpi błąd:
Wyjaśnij, którego pliku nie udało się zapisać i dlaczego.

Nie rób już nic więcej.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie zapisania `emocje.csv` i `emocje_postaci.csv`.
- Podgląd emocjonalnych portretów postaci.
- Wskazówkę jak pobrać pliki z Colab.

---
## Co dalej?

- Masz teraz `emocje.csv` (kwestie z walencją i emocją) i `emocje_postaci.csv` (emocjonalne portrety postaci).
- W **NB3** połączysz te dane z siecią z modułu 1 i wygenerujesz prezentację HTML z czterema wykresami.
- Jeśli łuk emocjonalny jest zbyt płaski albo rozkład emocji zdegenerowany, wróć do Kroku 3A i zmodyfikuj wbudowaną instrukcję oceny emocji — iteracja jest częścią metody.